##Analysis of Cursor Movement

#Creation of Data Frame for Basic reaching and Continuous tracking

#Basic Reaching

#Continuous Tracing

In [9]:
import pandas as pd
from pathlib import Path

# 1. Den Hauptordner definieren (das 'r' vor dem String ist wichtig für Windows-Pfade, damit Backslashes nicht als Steuerzeichen gelesen werden)
basis_ordner = Path("data/Continuous_tracking")

# Eine leere Liste, in der wir alle einzelnen Datensätze sammeln
alle_dfs = []

# 2. Automatisch alle CSV-Dateien suchen
# .rglob("*.csv") sucht rekursiv (also im Hauptordner UND in allen Unterordnern) nach CSV-Dateien
for datei_pfad in basis_ordner.rglob("*.csv"):
    
    # 3. CSV einlesen
    df_temp = pd.read_csv(datei_pfad)
    
    # WICHTIG: Probanden-ID aus dem Dateipfad extrahieren und als Spalte anfügen.
    # datei_pfad.parent.name gibt uns genau den Namen des Unterordners (z.B. 'ch51' oder 'ir31')
    df_temp['Participant'] = datei_pfad.parent.name
    
    # Den markierten DataFrame in unsere Liste packen
    alle_dfs.append(df_temp)

# 4. Alle Datensätze aus der Liste zu einem großen Master-DataFrame zusammenführen
# ignore_index=True sorgt dafür, dass die Zeilen von 0 bis X durchgehend neu nummeriert werden
df_tracking = pd.concat(alle_dfs, ignore_index=True)

# 5. Daten inspizieren
print(df_tracking.head())
print("\n" + "="*50 + "\n") # Nur ein optischer Trennstrich
print(df_tracking.info())

   target_width  cursor-width  target_pos_x  target_pos_y  cursor_pos_x  \
0           0.1           0.0           0.0           0.0      0.239135   
1           0.1           0.0           0.0           0.0      0.239135   
2           0.1           0.0           0.0           0.0      0.211002   
3           0.1           0.0           0.0           0.0      0.196935   
4           0.1           0.0           0.0           0.0      0.182868   

   cursor_pos_y  target_vel_x  target_vel_y  timestamp  trial_number  \
0      0.014067           0.0           0.0   0.000008             0   
1      0.014067           0.0           0.0   0.006560             0   
2      0.000000           0.0           0.0   0.012574             0   
3     -0.014067           0.0           0.0   0.019620             0   
4     -0.028134           0.0           0.0   0.027063             0   

   training  gaze_pos_x  gaze_pos_y  tracker_time  state_marker Participant  
0      True   33.900024   11.900024   

##BAsic Reaching

In [15]:
# 1. Den Hauptordner definieren 
basis_ordner = Path("data/Basic_reaching")

# Eine leere Liste, in der wir alle einzelnen Datensätze sammeln
alle_dfs = []

# 2. Automatisch alle CSV-Dateien suchen
for datei_pfad in basis_ordner.rglob("*.csv"):
    
    # 3. CSV einlesen
    df_temp = pd.read_csv(datei_pfad)
    
    # WICHTIG: Probanden-ID aus dem Dateipfad extrahieren und als Spalte anfügen.
    df_temp['Participant'] = datei_pfad.parent.name
    
    # ===============================================================
    # NEU: Hier fügen wir die Markierung für die Trainingsdaten ein!
    df_temp['is_training'] = 'block_0' in datei_pfad.name
    # ===============================================================
    
    # Den markierten DataFrame in unsere Liste packen
    alle_dfs.append(df_temp)

# 4. Alle Datensätze aus der Liste zu einem großen Master-DataFrame zusammenführen
df_reaching = pd.concat(alle_dfs, ignore_index=True)

# 5. Daten inspizieren (Hier habe ich df_tracking zu df_reaching korrigiert)
print(df_reaching.head())
print("\n" + "="*50 + "\n") 
print(df_reaching.info())

   trial      time  cursor_x_pix  cursor_y_pix  cursor_x_cm  cursor_y_cm  \
0      0  3.500525          48.0         106.0     1.119375     2.480694   
1      0  3.527586          48.0         106.0     1.119375     2.480694   
2      0  3.534337          48.0         106.0     1.119375     2.480694   
3      0  3.541171          48.0         106.0     1.119375     2.480694   
4      0  3.548086          48.0         106.0     1.119375     2.480694   

      gaze_x     gaze_y    target_x  target_y  state_marker  button_pressed  \
0  21.599976 -24.700012  515.424483 -52.85573             0           False   
1  23.400024 -20.299988  515.424483 -52.85573             0           False   
2  23.400024 -21.900024  515.424483 -52.85573             0           False   
3  23.300049 -20.299988  515.424483 -52.85573             0           False   
4  23.500000 -20.900024  515.424483 -52.85573             0           False   

   start_time Participant  is_training  
0    0.751293        CH51  

## Analysis

In [8]:
print(df_tracking['Participant'].unique())
print(df_reaching['Participant'].unique())

['ch51' 'ir31' 'lj25' 'pf13']
['CH51' 'IR31' 'LJ25' 'PF13']


## Cleaning data of training trials

Cleaning continous tracking

In [10]:
# Wir filtern den Master-DataFrame und speichern das Ergebnis in einer neuen Variable
# Wichtig: Bei Booleans (True/False) schreiben wir False ohne Anführungszeichen!
df_continuous_clean = df_tracking[df_tracking['training'] == False]

# Ein kurzer Check, um sicherzugehen, dass das Filtern geklappt hat:
# .unique() sollte uns jetzt nur noch [False] ausgeben.
print("Verbleibende Werte in der 'training' Spalte:")
print(df_continuous_clean['training'].unique())

# Optional: Schauen wir, wie viele Zeilen wir vorher und nachher haben
print(f"Zeilen vorher: {len(df_tracking)}")
print(f"Zeilen nachher: {len(df_continuous_clean)}")

Verbleibende Werte in der 'training' Spalte:
[False]
Zeilen vorher: 491512
Zeilen nachher: 472480


Cleaning basic reaching

In [21]:
# Wir filtern den Master-DataFrame und speichern das Ergebnis in einer neuen Variable
# Wichtig: Bei Booleans (True/False) schreiben wir False ohne Anführungszeichen!
df_reaching_clean = df_reaching[df_reaching['is_training'] == False]

# Ein kurzer Check, um sicherzugehen, dass das Filtern geklappt hat:
# .unique() sollte uns jetzt nur noch [False] ausgeben.
print("Verbleibende Werte in der 'training' Spalte:")
print(df_reaching_clean['is_training'].unique())

# Optional: Schauen wir, wie viele Zeilen wir vorher und nachher haben
print(f"Zeilen vorher: {len(df_reaching)}")
print(f"Zeilen nachher: {len(df_reaching_clean)}")

Verbleibende Werte in der 'training' Spalte:
[False]
Zeilen vorher: 243743
Zeilen nachher: 181697
